# Day4_01B_Hello_LangGraph_State_Nodes_Edges

## LangGraph - Your First Graph Workflow

**Duration:** 20–25 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Session:** Day 4 - LangGraph Hands-on  

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand the basic building blocks of LangGraph.
- Define a workflow **State**.
- Create simple **Nodes** as Python functions.
- Connect nodes using **Edges**.
- Use `START` and `END`.
- Compile and run a LangGraph workflow.
- Relate LangGraph to classroom and enterprise workflows.

---

## Previous Notebook Recap

In the previous notebook, we learned why LangGraph is useful.

CrewAI helps us build **Agent teams**.

LangGraph helps us build **stateful workflows**.

Today we will build our first LangGraph workflow.


# 1. What Are We Building?

We will build a simple DSU teaching workflow.

```text
START
  ↓
Create Lesson Plan
  ↓
Create Summary
  ↓
END
```

This is intentionally simple.

The goal is to understand:

- State
- Node
- Edge
- START
- END
- Graph compilation
- Graph execution


# 2. LangGraph Mental Model

LangGraph has three core ideas.

| Concept | Simple Meaning |
|---|---|
| State | Shared data moving through the workflow |
| Node | A step that reads and updates state |
| Edge | A connection between steps |

---

## Simple Analogy

Think of a college approval file.

The file moves from one desk to another.

Each person adds something to the file.

```text
File → Faculty → Reviewer → HOD → Final
```

In LangGraph:

- File = State
- Faculty/Reviewer/HOD = Nodes
- Movement between desks = Edges


# 3. Setup

Install dependencies if needed:

```bash
pip install langgraph langchain-openai python-dotenv
```

We will keep the first demo mostly Python-based so faculty can understand the workflow before adding complex LLM calls.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 4. Define the State

State is the shared information that moves through the graph.

For our teaching workflow, the state will contain:

- topic
- lesson_plan
- summary

We use `TypedDict` to clearly define the structure.


In [ ]:
class TeachingState(TypedDict):
    topic: str
    lesson_plan: str
    summary: str

## What did we just learn?

State is like a shared notebook.

Each node can read from it and write back to it.

At the start:

```python
{
    "topic": "Agentic AI",
    "lesson_plan": "",
    "summary": ""
}
```

At the end, it will contain the completed workflow output.


# 5. Create Node 1 - Lesson Plan Node

A node is simply a Python function.

It receives the state.

It returns updates to the state.


In [ ]:
def create_lesson_plan(state: TeachingState):
    topic = state["topic"]

    lesson_plan = f"""
    Lesson Plan for: {topic}

    1. Start with a simple real-world question.
    2. Explain the core concept in simple terms.
    3. Show one classroom-friendly example.
    4. Discuss one enterprise use case.
    5. End with a short recap question.
    """

    return {
        "lesson_plan": lesson_plan
    }

# 6. Create Node 2 - Summary Node

This node uses the lesson plan from the state and creates a summary.


In [ ]:
def create_summary(state: TeachingState):
    topic = state["topic"]
    lesson_plan = state["lesson_plan"]

    summary = f"""
    Summary for {topic}:

    This lesson introduces {topic} using a simple classroom question,
    a concept explanation, one practical example, and one enterprise use case.

    The session can end with a recap question to check understanding.
    """

    return {
        "summary": summary
    }

## Pause & Reflect

Notice something important.

We did not call `create_summary()` directly after `create_lesson_plan()`.

LangGraph will manage the flow using edges.

That is the graph idea.


# 7. Create the Graph Builder

Now we create a graph using the state structure.


In [ ]:
graph_builder = StateGraph(TeachingState)

# 8. Add Nodes to the Graph

We now register the Python functions as graph nodes.


In [ ]:
graph_builder.add_node("create_lesson_plan", create_lesson_plan)
graph_builder.add_node("create_summary", create_summary)

# 9. Add Edges

Edges define the workflow path.

Our path is:

```text
START → create_lesson_plan → create_summary → END
```


In [ ]:
graph_builder.add_edge(START, "create_lesson_plan")
graph_builder.add_edge("create_lesson_plan", "create_summary")
graph_builder.add_edge("create_summary", END)

# 10. Compile the Graph

Before running, we compile the graph.

Compilation checks and prepares the workflow.


In [ ]:
teaching_graph = graph_builder.compile()

print("Graph compiled successfully.")

# 11. Run the Graph

Now we pass the initial state.

LangGraph runs the workflow and returns the final state.


In [ ]:
initial_state = {
    "topic": "Agentic AI",
    "lesson_plan": "",
    "summary": ""
}

final_state = teaching_graph.invoke(initial_state)

final_state

# 12. Display the Output Cleanly

In [ ]:
print("=" * 80)
print("LANGGRAPH TEACHING WORKFLOW OUTPUT")
print("=" * 80)

print("\nTOPIC")
print("-" * 80)
print(final_state["topic"])

print("\nLESSON PLAN")
print("-" * 80)
print(final_state["lesson_plan"])

print("\nSUMMARY")
print("-" * 80)
print(final_state["summary"])

# 13. What Did We Build?

We built our first LangGraph workflow.

```text
Input State
   ↓
create_lesson_plan node
   ↓
Updated State
   ↓
create_summary node
   ↓
Final State
```

The graph controlled the flow.

Each node updated the shared state.


# 14. Modify the Topic

Change the topic and run again.


In [ ]:
initial_state = {
    "topic": "Retrieval Augmented Generation",
    "lesson_plan": "",
    "summary": ""
}

final_state = teaching_graph.invoke(initial_state)

print(final_state["lesson_plan"])
print("\n--- SUMMARY ---\n")
print(final_state["summary"])

# 15. Add a Third Node

Let us extend the workflow by adding a quiz node.

New flow:

```text
START
  ↓
Create Lesson Plan
  ↓
Create Summary
  ↓
Create Quiz
  ↓
END
```


First, define a new state that includes quiz.


In [ ]:
class TeachingStateWithQuiz(TypedDict):
    topic: str
    lesson_plan: str
    summary: str
    quiz: str

In [ ]:
def create_quiz(state: TeachingStateWithQuiz):
    topic = state["topic"]

    quiz = f"""
    Quiz for {topic}

    1. What is the main idea of {topic}?
    2. Give one classroom example of {topic}.
    3. Give one enterprise use case of {topic}.
    """

    return {
        "quiz": quiz
    }

# 16. Build the Extended Graph

In [ ]:
extended_builder = StateGraph(TeachingStateWithQuiz)

extended_builder.add_node("create_lesson_plan", create_lesson_plan)
extended_builder.add_node("create_summary", create_summary)
extended_builder.add_node("create_quiz", create_quiz)

extended_builder.add_edge(START, "create_lesson_plan")
extended_builder.add_edge("create_lesson_plan", "create_summary")
extended_builder.add_edge("create_summary", "create_quiz")
extended_builder.add_edge("create_quiz", END)

extended_graph = extended_builder.compile()

print("Extended graph compiled successfully.")

# 17. Run the Extended Graph

In [ ]:
initial_state = {
    "topic": "Function Tools in Agentic AI",
    "lesson_plan": "",
    "summary": "",
    "quiz": ""
}

final_state = extended_graph.invoke(initial_state)

print(final_state["lesson_plan"])
print("\n--- SUMMARY ---\n")
print(final_state["summary"])
print("\n--- QUIZ ---\n")
print(final_state["quiz"])

# 18. Enterprise Connection

This same structure applies to enterprise workflows.

## Example: Incident Report Workflow

```text
START
  ↓
Collect Incident Details
  ↓
Analyze Logs
  ↓
Create RCA Summary
  ↓
END
```

State may contain:

```python
{
    "incident_id": "...",
    "logs": "...",
    "analysis": "...",
    "rca_summary": "..."
}
```

LangGraph gives control over how each step updates the workflow data.


# 19. Classroom Exercise

Create a graph for:

> Student Assignment Preparation

Workflow:

```text
START
  ↓
Create Assignment
  ↓
Create Rubric
  ↓
Create Submission Instructions
  ↓
END
```

State should contain:

- topic
- assignment
- rubric
- instructions


In [ ]:
# Exercise Starter Code

class AssignmentState(TypedDict):
    topic: str
    assignment: str
    rubric: str
    instructions: str

# TODO:
# 1. Create create_assignment node
# 2. Create create_rubric node
# 3. Create create_instructions node
# 4. Build StateGraph
# 5. Add nodes and edges
# 6. Compile and invoke the graph


# 20. Key Takeaways

In this notebook, we learned:

- LangGraph workflows are made of state, nodes, and edges.
- State is the shared data moving through the graph.
- A node is a Python function that updates state.
- Edges define the workflow path.
- `START` and `END` define the graph boundaries.
- `compile()` prepares the graph.
- `invoke()` runs the graph.

---

## Final Mental Model

```text
State enters graph
   ↓
Node updates state
   ↓
Edge moves to next node
   ↓
Final state comes out
```

Now we are ready to add conditional routing and loops.
